In [7]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import xmltodict, json
import ast
import numbers
import shlex # package to construct the git command to subprocess format
import subprocess 
%matplotlib inline

In [8]:
def findModel(Parent,PathElements):
    for pe in PathElements:
        Parent = findNextChild(Parent,pe)
    return Parent

def findNextChild(Parent,ChildName):
    if len(Parent['Children']) >0:
        for child in range(len(Parent['Children'])):
            if Parent['Children'][child]['Name'] == ChildName:
                return Parent['Children'][child]
    else:
        return Parent[ChildName]
   
def replaceModel(Parent,modelPath,New):
    PathElements = modelPath.split('.')
    try:
        test = findModel(Parent,PathElements[:-1])[PathElements[-1]]
        findModel(Parent,PathElements[:-1])[PathElements[-1]] = New
    except:
        try:
            pos = 0
            for kid in findModel(Parent,PathElements[:-1])['Children']:
                if kid['Name'] == PathElements[-1]:
                    findModel(Parent,PathElements[:-1])['Children'][pos] = New
                    break
                pos +=1
        except:   
            print('Could not find parent node of model to over write for ' + modelPath)
            raise

In [9]:
# command= "git --git-dir=C:/GitHubRepos/ApsimX/.git --work-tree=C:/GitHubRepos/ApsimX checkout upstream/master C:/GitHubRepos/ApsimX/Tests/Validation/Wheat/Wheat.apsimx" 
# #command= "git --git-dir=C:/GitHubRepos/ApsimX/.git --work-tree=C:/GitHubRepos/ApsimX checkout C:/GitHubRepos/ApsimX/Models/Resources/Wheat.json" 
# comm=shlex.split(command) # This will convert the command into list format
# subprocess.run(comm, shell=True) # Run the git command

In [10]:
## Read wheat test file into json object
with open('C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Wheat.apsimx','r') as WheatTestsJSON:
    WheatTests = json.load(WheatTestsJSON)
    WheatTestsJSON.close()
    ## read prototype wheat file into json object
with open('C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatPrototype.apsimx','r') as WheatPrototypeJSON:
    WheatPrototype = json.load(WheatPrototypeJSON)
    WheatPrototypeJSON.close()

In [11]:
#Copy prototype wheat model out of replacements and put it in replacements in test file
Replacements =  findModel(WheatPrototype,['Replacements'])
replaceModel(WheatTests,'Replacements',Replacements)

In [12]:
with open('C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatSL.apsimx','w') as WheatTestsJSON:
    json.dump(WheatTests ,WheatTestsJSON,indent=2)

In [13]:
replacements = pd.read_excel('C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\SimpleLeafImplementation\VariableRenames.xlsx',index_col=0).to_dict()['SimpleLeaf']
with open(r'C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatSL.apsimx', 'r') as file: 
    data = file.read() 
    for v in replacements.keys():
        data = data.replace(v, replacements[v])
        w = v.replace('Wheat','[Wheat]')
        rw = replacements[v].replace('Wheat','[Wheat]')
        data = data.replace(w, rw)
        
# Opening our text file in write only 
# mode to write the replaced content 
with open(r'C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatSL.apsimx', 'w') as file: 
  
    # Writing the replaced data in our 
    # text file 
    file.write(data) 

In [37]:
#replacements = pd.read_excel('C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\SimpleLeafImplementation\VariableRenames.xlsx',index_col=0).to_dict()['SimpleLeaf']
replacements = pd.read_excel('C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\SimpleLeafImplementation\VariableRenames.xlsx',index_col=0,sheet_name='Existing Model renames').to_dict()['SimpleLeaf']

from pathlib import Path
fileLoc = 'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\data'
Allcols = []
pathlist = Path(fileLoc).glob('**/*.xlsx')
for path in pathlist:
    # because path is object not string
    obsDat = pd.read_excel(path, engine='openpyxl',sheet_name='Observed')
    newCols = []
    for c in obsDat.columns:
        Allcols.append(c)
        if c in replacements.keys():
            newCols.append(c.replace(c,replacements[c]))
        else:
            newCols.append(c)
    obsDat.columns = newCols
    with pd.ExcelWriter(path, engine='openpyxl', mode='a',if_sheet_exists='replace') as writer: 
        workbook = writer.book
        obsDat.to_excel(writer,index=False,sheet_name='Observed')

In [15]:
for path in pathlist:
    # because path is object not string
    obsDat = pd.read_excel(path, engine='openpyxl',sheet_name='MaxLeafSize')
    newCols = []
    for c in obsDat.columns:
        if c in replacements.keys():
            newCols.append(c.replace(c,replacements[c]))
        else:
            newCols.append(c)
    obsDat.columns = newCols
    with pd.ExcelWriter(path, engine='openpyxl', mode='a',if_sheet_exists='replace') as writer: 
        workbook = writer.book
        obsDat.to_excel(writer,index=False,sheet_name='MaxLeafSize')

In [19]:
AllObs = list(set(Allcols))

In [28]:
AllObs[0]

'Wheat.Stem.N'

In [30]:
AllObs[0].replace(AllObs[0].split('.')[0],"["+AllObs[0].split('.')[0]+"]" )

'[Wheat].Stem.N'

In [31]:
[x.replace(x.split('.')[0],"["+x.split('.')[0]+"]") for x in AllObs]

['[Wheat].Stem.N',
 '[Wheat].Ear.Nconc',
 '[Soil].Water.Volumetric(4)',
 '[Soil].Water.Volumetric(6)',
 '[Wheat].Leaf.Dead.N',
 '[Soil].Water.Volumetric(3)',
 '[Wheat].Leaf.Live.NError',
 '[Wheat].Leaf.CoverTotalError',
 '[Wheat].Phenology.ReadyForHarvestDAS',
 '[ObservedLayers].SW(4)Error',
 '[Wheat].Leaf.Height.se',
 '[Wheat].SowingData.Cultivar',
 '[ObservedLayers].SW(7)',
 '[Wheat].SowingData.Population',
 '[Wheat].Leaf.Tips',
 '[ObservedLayers].SW(2)',
 '[Wheat].Phenology.FloweringDAS',
 '[Wheat].Grain.Density',
 '[Clock].Today',
 '[Wheat].AboveGround.Nconc.se',
 '[NDVIModel].Script.NDVI.se',
 '[Wheat].Grain.WtError',
 '[Wheat].Leaf.StemNumberPerPlant',
 '[Wheat].Leaf.Live.Nconc',
 '[Wheat].Leaf.SpecificArea',
 '[ObservedLayers].SW(10)',
 '[NDVIModel].Script.NDVIError',
 '[Wheat].Structure.TotalStemPopn',
 '[ObservedLayers].SW(8)',
 '[Wheat].Leaf.DeadCohortNo',
 '[Wheat].Leaf.Live.Wt',
 '[Wheat].Phenology.MaturityDASError',
 '[SimulationName]',
 '[Wheat].Phenology.FinalLeafNumberE

In [16]:
a = list(set(Allcols+list(replacements.keys())))
a.sort()

In [17]:
a

['([Wheat].Leaf.Transpiration + [Soil].SoilWater.Es + [MicroClimate].PrecipitationInterception)',
 'Clock.Today',
 'NDVIModel.Script.NDVI',
 'NDVIModel.Script.NDVI.se',
 'NDVIModel.Script.NDVIError',
 'Notes',
 'ObservedLayers.SW(1)',
 'ObservedLayers.SW(1)Error',
 'ObservedLayers.SW(10)',
 'ObservedLayers.SW(10)Error',
 'ObservedLayers.SW(2)',
 'ObservedLayers.SW(2)Error',
 'ObservedLayers.SW(3)',
 'ObservedLayers.SW(3)Error',
 'ObservedLayers.SW(4)',
 'ObservedLayers.SW(4)Error',
 'ObservedLayers.SW(5)',
 'ObservedLayers.SW(5)Error',
 'ObservedLayers.SW(6)',
 'ObservedLayers.SW(6)Error',
 'ObservedLayers.SW(7)',
 'ObservedLayers.SW(7)Error',
 'ObservedLayers.SW(8)',
 'ObservedLayers.SW(8)Error',
 'ObservedLayers.SW(9)',
 'ObservedLayers.SW(9)Error',
 'ProfileWater',
 'SimulationName',
 'Soil.Water.Volumetric(1)',
 'Soil.Water.Volumetric(10)',
 'Soil.Water.Volumetric(2)',
 'Soil.Water.Volumetric(3)',
 'Soil.Water.Volumetric(4)',
 'Soil.Water.Volumetric(5)',
 'Soil.Water.Volumetric(6)'